## Instalasi & Import Library

In [ ]:
!pip install kagglehub --quiet

import kagglehub, os, shutil, random, json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

print(f"TensorFlow version : {tf.__version__}")
print(f"GPU available      : {tf.config.list_physical_devices('GPU')}")
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)


## Download & Susun Dataset

In [ ]:
path = kagglehub.dataset_download("shanks0465/braille-character-dataset")
print("Path to downloaded dataset:", path)

DATASET_ROOT = Path('/content/dataset')
if DATASET_ROOT.exists():
    shutil.rmtree(DATASET_ROOT)
DATASET_ROOT.mkdir(parents=True)

source_path = Path(path)
images = list(source_path.rglob('*.png')) + list(source_path.rglob('*.jpg'))

for img in images:
    char = img.stem[0].lower()
    if 'a' <= char <= 'z':
        char_dir = DATASET_ROOT / char
        char_dir.mkdir(exist_ok=True)
        shutil.copy(img, char_dir / img.name)

print(f"\nDataset disusun di {DATASET_ROOT}")
for item in sorted(DATASET_ROOT.iterdir()):
    print(f"  {item.name.upper()}: {len(list(item.glob('*.*')))} file")


## Konfigurasi & Persiapan Data

In [ ]:
IMG_SIZE    = 64
BATCH_SIZE  = 32
NUM_CLASSES = 26
EPOCHS      = 100
LR          = 1e-3

all_paths, all_labels = [], []
subdirs = sorted([d for d in DATASET_ROOT.iterdir() if d.is_dir()])

for cls_dir in subdirs:
    char = cls_dir.name
    imgs = list(cls_dir.glob('*.png')) + list(cls_dir.glob('*.jpg'))
    for img in imgs:
        all_paths.append(str(img))
        all_labels.append(char)

unique_classes = sorted(set(all_labels))
class_to_idx  = {cls: i for i, cls in enumerate(unique_classes)}
idx_to_class  = {i: cls for cls, i in class_to_idx.items()}
int_labels    = [class_to_idx[l] for l in all_labels]

print(f"Total gambar    : {len(all_paths)}")
print(f"Total kelas     : {len(unique_classes)}")

train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    all_paths, int_labels, test_size=0.30, random_state=SEED, stratify=int_labels
)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.50, random_state=SEED, stratify=temp_labels
)
print(f"   Train : {len(train_paths)} | Val : {len(val_paths)} | Test : {len(test_paths)}")


## tf.data Pipeline & Augmentasi

In [ ]:
def load_and_preprocess(img_path, label):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_image(img, channels=1, expand_animations=False)  # Grayscale
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE], method='nearest')     # Titik tetap tajam
    img = tf.cast(img, tf.float32) / 255.0                                 # Normalisasi [0,1]
    img = tf.image.grayscale_to_rgb(img)                                   # Expand ke 3ch untuk CNN
    return img, label

augmentation = keras.Sequential([
    layers.RandomRotation(0.03),           # ±11 derajat
    layers.RandomZoom((-0.1, 0.1)),        # ±10%
    layers.RandomTranslation(0.05, 0.05),  # ±5%
    layers.RandomContrast(0.2),            # Variasi kontras
], name='augmentation')

AUTOTUNE = tf.data.AUTOTUNE

train_ds = (tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
            .shuffle(len(train_paths), seed=SEED)
            .map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
            .map(lambda x, y: (augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
            .batch(BATCH_SIZE)
            .prefetch(AUTOTUNE))

val_ds   = (tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
            .map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
            .batch(BATCH_SIZE)
            .prefetch(AUTOTUNE))

test_ds  = (tf.data.Dataset.from_tensor_slices((test_paths, test_labels))
            .map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
            .batch(BATCH_SIZE)
            .prefetch(AUTOTUNE))

sample_batch = next(iter(train_ds))
print(f"Pipeline siap! Batch shape: {sample_batch[0].shape}")

fig, axes = plt.subplots(2, 8, figsize=(18, 5))
fig.suptitle('Sample Gambar (Atas: Asli | Bawah: Setelah Augmentasi)', fontsize=13, fontweight='bold')
for i in range(8):
    raw_img, lbl = load_and_preprocess(train_paths[i], train_labels[i])
    aug_img = augmentation(raw_img, training=True)
    axes[0][i].imshow(raw_img.numpy(), cmap='gray'); axes[0][i].axis('off')
    axes[0][i].set_title(idx_to_class[train_labels[i].numpy() if hasattr(train_labels[i], 'numpy') else train_labels[i]].upper(), fontsize=11)
    axes[1][i].imshow(aug_img.numpy(), cmap='gray'); axes[1][i].axis('off')
plt.tight_layout(); plt.show()


## Custom CNN



In [ ]:
def build_braille_cnn(num_classes=26, img_size=64):
    """
    Custom CNN ringan untuk klasifikasi karakter Braille.
    Arsitektur: Conv-BN-ReLU blok bertingkat + GlobalAveragePooling + Dense head
    Input: (batch, 64, 64, 3)  →  Output: (batch, 26)
    """
    inputs = keras.Input(shape=(img_size, img_size, 3), name='input')

    # Blok 1: 64 filter, stride 1
    x = layers.Conv2D(32, 3, padding='same', use_bias=False, name='conv1a')(inputs)
    x = layers.BatchNormalization(name='bn1a')(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(32, 3, padding='same', use_bias=False, name='conv1b')(x)
    x = layers.BatchNormalization(name='bn1b')(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPool2D(2, name='pool1')(x)   # 32x32
    x = layers.Dropout(0.15)(x)

    # Blok 2
    x = layers.Conv2D(64, 3, padding='same', use_bias=False, name='conv2a')(x)
    x = layers.BatchNormalization(name='bn2a')(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(64, 3, padding='same', use_bias=False, name='conv2b')(x)
    x = layers.BatchNormalization(name='bn2b')(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPool2D(2, name='pool2')(x)   # 16x16
    x = layers.Dropout(0.2)(x)

    # Blok 3
    x = layers.Conv2D(128, 3, padding='same', use_bias=False, name='conv3a')(x)
    x = layers.BatchNormalization(name='bn3a')(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(128, 3, padding='same', use_bias=False, name='conv3b')(x)
    x = layers.BatchNormalization(name='bn3b')(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPool2D(2, name='pool3')(x)   # 8x8
    x = layers.Dropout(0.25)(x)

    # Blok 4
    x = layers.Conv2D(256, 3, padding='same', use_bias=False, name='conv4a')(x)
    x = layers.BatchNormalization(name='bn4a')(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPool2D(2, name='pool4')(x)   # 4x4
    x = layers.Dropout(0.3)(x)

    # Classifier Head
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dense(256, use_bias=False, name='fc1')(x)
    x = layers.BatchNormalization(name='bn_fc1')(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='output')(x)

    model = keras.Model(inputs, outputs, name='BrailleVision_CNN')
    return model

model = build_braille_cnn(NUM_CLASSES, IMG_SIZE)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR),
    loss='sparse_categorical_crossentropy',
    metrics=[
        'accuracy',
        keras.metrics.SparseTopKCategoricalAccuracy(k=3, name='top3_acc')
    ]
)

model.summary()
print(f"\nTotal params: {model.count_params():,}")


---
## Training Model

In [ ]:
os.makedirs('/content/checkpoints', exist_ok=True)
checkpoint_path = '/content/checkpoints/braille_cnn_best.keras'

callbacks = [
    ModelCheckpoint(
        checkpoint_path,
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy',
        patience=20,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=8,
        min_lr=1e-7,
        verbose=1
    )
]

print("Memulai Training Custom CNN (dari nol)")
print(f"   Input : {IMG_SIZE}×{IMG_SIZE}px grayscale → 3ch")
print(f"   Epochs: hingga {EPOCHS} (EarlyStopping aktif)")
print("=" * 60)

history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    callbacks=callbacks,
    verbose=1
)

best_val = max(history.history['val_accuracy'])
print(f"\nTraining selesai! Best Val Accuracy: {best_val*100:.2f}%")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Training History — BrailleVision Custom CNN', fontsize=14, fontweight='bold')

h = history.history
ep = range(len(h['accuracy']))

axes[0].plot(ep, h['accuracy'],     label='Train', color='steelblue')
axes[0].plot(ep, h['val_accuracy'], label='Val',   color='orangered')
axes[0].axhline(0.85, color='green', linestyle='--', alpha=0.7, label='Target 85%')
axes[0].set_title('Accuracy'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ep, h['loss'],     label='Train', color='steelblue')
axes[1].plot(ep, h['val_loss'], label='Val',   color='orangered')
axes[1].set_title('Loss'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(ep, h['top3_acc'],     label='Train Top-3', color='steelblue')
axes[2].plot(ep, h['val_top3_acc'], label='Val Top-3',   color='orangered')
axes[2].set_title('Top-3 Accuracy'); axes[2].set_xlabel('Epoch'); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout(); plt.show()



## Evaluasi Model: Test Set & Confusion Matrix

In [ ]:
print("Evaluasi pada Test Set...")
test_loss, test_acc, test_top3 = model.evaluate(test_ds, verbose=0)

print(f"\n{'='*50}")
print(f"  Test Loss     : {test_loss:.4f}")
print(f"  Test Accuracy : {test_acc*100:.2f}%")
print(f"  Test Top-3 Acc: {test_top3*100:.2f}%")
print(f"{'='*50}")

if test_acc >= 0.85:
    print("\nTarget akurasi ≥ 85% TERCAPAI! 🎉")
else:
    print(f"\nAkurasi {test_acc*100:.2f}% belum mencapai target 85%.")

# Confusion Matrix
y_true, y_pred_probs = [], []
for imgs, labels in test_ds:
    preds = model(imgs, training=False)
    y_true.extend(labels.numpy())
    y_pred_probs.extend(preds.numpy())

y_pred = np.argmax(y_pred_probs, axis=1)
class_name_labels = [idx_to_class[i].upper() for i in range(NUM_CLASSES)]

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_name_labels))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_name_labels, yticklabels=class_name_labels,
            linewidths=0.5, ax=ax)
ax.set_title(f'Confusion Matrix — Test Set (Acc: {test_acc*100:.2f}%)', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Confusion matrix tersimpan ke /content/confusion_matrix.png")


In [ ]:
per_class_acc = cm.diagonal() / cm.sum(axis=1)
fig, ax = plt.subplots(figsize=(16, 5))
bars = ax.bar(class_name_labels, per_class_acc * 100, color=[
    'green' if a >= 0.85 else 'orange' if a >= 0.70 else 'red'
    for a in per_class_acc
])
ax.axhline(85, color='red', linestyle='--', label='Target 85%')
ax.set_title('Akurasi per Kelas (Braille A–Z)', fontsize=13, fontweight='bold')
ax.set_xlabel('Kelas'); ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 110); ax.legend()

for bar, acc in zip(bars, per_class_acc):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{acc*100:.0f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout(); plt.show()

print("\nKelas dengan akurasi < 85% (perlu perhatian):")
for cls, acc in zip(class_name_labels, per_class_acc):
    if acc < 0.85:
        print(f"   {cls}: {acc*100:.1f}%")


## Simpan Model & Konversi ke TFLite

In [ ]:
SAVE_DIR = '/content/braille_model'
os.makedirs(SAVE_DIR, exist_ok=True)

# Simpan model Keras
model_keras_path = f'{SAVE_DIR}/braille_vision_model.keras'
model.save(model_keras_path)
size_mb = os.path.getsize(model_keras_path) / (1024**2)
print(f"Model Keras tersimpan: {model_keras_path} ({size_mb:.2f} MB)")

# Simpan label map
label_map = {str(i): cls.upper() for i, cls in idx_to_class.items()}
with open(f'{SAVE_DIR}/label_map.json', 'w') as f:
    json.dump(label_map, f, indent=2)
print(f"Label map tersimpan.")

# TFLite Float16
print("\nKonversi ke TFLite Float16...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_f16 = converter.convert()
tflite_f16_path = f'{SAVE_DIR}/braille_vision_f16.tflite'
with open(tflite_f16_path, 'wb') as f:
    f.write(tflite_f16)
size_f16 = os.path.getsize(tflite_f16_path) / (1024**2)
print(f"TFLite Float16: {size_f16:.2f} MB")

# TFLite Int8 (terkecil, untuk perangkat edge)
print("\nKonversi ke TFLite Int8...")
def representative_dataset():
    for imgs, _ in train_ds.take(50):
        for img in imgs:
            yield [tf.expand_dims(img, 0)]

converter2 = tf.lite.TFLiteConverter.from_keras_model(model)
converter2.optimizations = [tf.lite.Optimize.DEFAULT]
converter2.representative_dataset = representative_dataset
converter2.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter2.inference_input_type  = tf.int8
converter2.inference_output_type = tf.int8
tflite_int8 = converter2.convert()
tflite_int8_path = f'{SAVE_DIR}/braille_vision_int8.tflite'
with open(tflite_int8_path, 'wb') as f:
    f.write(tflite_int8)
size_int8 = os.path.getsize(tflite_int8_path) / (1024**2)
print(f"TFLite Int8: {size_int8:.2f} MB")

print(f"\n{'='*55}")
print(f"Ringkasan Ukuran Model:")
print(f"     Keras original : {size_mb:.2f} MB")
print(f"     TFLite Float16 : {size_f16:.2f} MB  {'OK' if size_f16 < 5 else '>5MB'}")
print(f"     TFLite Int8    : {size_int8:.2f} MB  {'OK' if size_int8 < 5 else '>5MB'}")
print(f"{'='*55}")



## Validasi Model TFLite

In [ ]:
print("Memvalidasi akurasi TFLite Float16 pada test set...")

interpreter = tf.lite.Interpreter(model_path=tflite_f16_path)
interpreter.allocate_tensors()
input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

correct, total = 0, 0
for imgs, labels in test_ds:
    for img, label in zip(imgs, labels):
        inp = tf.expand_dims(img, 0)
        interpreter.set_tensor(input_details[0]['index'], inp)
        interpreter.invoke()
        pred = np.argmax(interpreter.get_tensor(output_details[0]['index']))
        if pred == label.numpy():
            correct += 1
        total += 1

tflite_acc = correct / total
print(f"\n{'='*50}")
print(f"  TFLite Float16 Accuracy : {tflite_acc*100:.2f}%")
print(f"  Keras model   Accuracy  : {test_acc*100:.2f}%")
print(f"  Delta                   : {abs(test_acc - tflite_acc)*100:.2f}%")
print(f"{'='*50}")

# Download file model
from google.colab import files
print("\n⬇Mengunduh model ke komputer Anda...")
files.download(tflite_f16_path)
files.download(tflite_int8_path)
files.download(f'{SAVE_DIR}/label_map.json')
print("Selesai! File model sudah diunduh.")
